# TurAIsta Safety Layer and Pivot Translation Test

The first evaluation found that NLLB-200 mistranslates allergens (peanuts became seeds, apples, or vegetables). This notebook tests two responses:

1. A rule-based safety detector that flags allergy, health, child, accessibility, and weather messages before translation. We measure how often it fires on ordinary FLORES news sentences (false alarms) and whether it catches realistic safety messages.
2. Pivot translation, which routes Korean, Japanese, and Chinese through English before translating into the host language. We compare it with direct translation on FLORES+ and on allergen term preservation.

FLORES+ devtest is used for evaluation only.

## Setup Checklist

1. Session options: Accelerator GPU T4 x1, Internet On.
2. Add-ons, Secrets: attach `HF_TOKEN`.
3. Run All. Expected runtime is 20 to 40 minutes.

## 1. Install and Configure

In [ ]:
!pip install -q sacrebleu

In [ ]:
import os, json, torch, sacrebleu
from pathlib import Path
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Turn on the GPU in Session options."
OUT = Path("/kaggle/working/safety_out"); OUT.mkdir(exist_ok=True)
N_MT = 100
RESULTS = {}
def save(): (OUT / "results.json").write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2))

## 2. Safety Bank and Detector

This cell defines the same module as `model_tools/safety/bank.py` in the repository, so the notebook runs without uploads.

In [ ]:
"""TurAIsta safety phrase bank and rule-based DIETARY_SAFETY detector.

Why this exists: the 2026-10-04 evaluation showed NLLB-200 mistranslating
"allergic to peanuts" as seeds (tl), apples (ceb) and vegetables (ilo). Safety
messages therefore never go through free translation. Instead:
  1. detect() scans the guest's message for allergen and risk keywords in
     en, ko, ja, zh. It is deliberately high-recall and runs BEFORE the intent model.
  2. Any hit forces intent DIETARY_SAFETY, whatever the model says.
  3. card() renders a fixed host-language card from verified terms only.

Every host-language term is a DRAFT until a native speaker sets reviewed=True.
card() refuses to show an unreviewed term without a visible warning.
"""

import re
import unicodedata

# confidence: "high" = standard everyday word; "check" = plausible, needs a native speaker.
# Guest-side keywords are matched as substrings (CJK) or word prefixes (en).
ITEMS = {
    # ---- allergens
    "peanut": {"kind": "allergen",
               "guest": {"en": ["peanut", "groundnut"], "ko": ["땅콩"], "ja": ["ピーナッツ", "落花生", "ピーナツ"], "zh": ["花生"]},
               "host": {"tl": ("mani", "high"), "ceb": ("mani", "high"), "ilo": ("mani", "high")}},
    "tree_nut": {"kind": "allergen",
                 "guest": {"en": ["tree nut", "nuts", "almond", "cashew", "walnut"], "ko": ["견과", "아몬드", "호두", "캐슈"],
                           "ja": ["ナッツ", "アーモンド", "くるみ", "クルミ", "カシュー"], "zh": ["坚果", "堅果", "杏仁", "核桃", "腰果"]},
                 "host": {"tl": ("mga nuwes (kasoy, almendras)", "check"), "ceb": ("mga nuts (kasoy, almendras)", "check"),
                          "ilo": ("mga nuts (kasoy)", "check")}},
    "shellfish": {"kind": "allergen",
                  "guest": {"en": ["shellfish", "shrimp", "prawn", "crab", "lobster", "crustacean"], "ko": ["갑각류", "새우", "게", "조개", "랍스터"],
                            "ja": ["甲殻類", "えび", "エビ", "海老", "かに", "カニ", "蟹", "貝"], "zh": ["甲壳", "甲殼", "虾", "蝦", "蟹", "贝", "貝"]},
                  "host": {"tl": ("hipon, alimango, alimasag, at iba pang shellfish", "high"),
                           "ceb": ("pasayan, kasag, alimango, kinhason", "check"),
                           "ilo": ("padaw, kappi, ken dadduma pay a shellfish", "check")}},
    "fish": {"kind": "allergen",
             "guest": {"en": ["fish"], "ko": ["생선", "물고기"], "ja": ["魚", "さかな"], "zh": ["鱼", "魚"]},
             "host": {"tl": ("isda", "high"), "ceb": ("isda", "high"), "ilo": ("ikan", "check")}},
    "egg": {"kind": "allergen",
            "guest": {"en": ["egg"], "ko": ["달걀", "계란"], "ja": ["卵", "たまご", "玉子"], "zh": ["鸡蛋", "雞蛋", "蛋"]},
            "host": {"tl": ("itlog", "high"), "ceb": ("itlog", "high"), "ilo": ("itlog", "high")}},
    "milk": {"kind": "allergen",
             "guest": {"en": ["milk", "dairy", "lactose", "cheese", "butter"], "ko": ["우유", "유제품", "유당", "치즈", "버터"],
                       "ja": ["牛乳", "乳製品", "乳糖", "チーズ", "バター", "ミルク"], "zh": ["牛奶", "奶制品", "乳制品", "乳糖", "奶酪", "黄油"]},
             "host": {"tl": ("gatas at mga produktong gawa sa gatas", "high"), "ceb": ("gatas ug mga produkto sa gatas", "check"),
                      "ilo": ("gatas ken dagiti produkto ti gatas", "check")}},
    "wheat_gluten": {"kind": "allergen",
                     "guest": {"en": ["wheat", "gluten", "celiac", "coeliac"], "ko": ["밀", "글루텐"], "ja": ["小麦", "グルテン"], "zh": ["小麦", "麸质", "麩質", "面筋"]},
                     "host": {"tl": ("trigo / harina (gluten)", "check"), "ceb": ("trigo / harina (gluten)", "check"),
                              "ilo": ("trigo / harina (gluten)", "check")}},
    "soy": {"kind": "allergen",
            "guest": {"en": ["soy", "soya", "tofu"], "ko": ["대두", "콩", "두부"], "ja": ["大豆", "豆腐", "醤油"], "zh": ["大豆", "黄豆", "豆腐", "酱油"]},
            "host": {"tl": ("soya / toyo / tokwa", "check"), "ceb": ("soya / toyo / tokwa", "check"), "ilo": ("soya / toyo / tokwa", "check")}},
    "sesame": {"kind": "allergen",
               "guest": {"en": ["sesame"], "ko": ["참깨", "깨"], "ja": ["ごま", "ゴマ", "胡麻"], "zh": ["芝麻"]},
               "host": {"tl": ("linga", "high"), "ceb": ("linga", "check"), "ilo": ("linga", "check")}},
    # ---- diets (the card asks the host to confirm ingredients, it never promises safety)
    "vegetarian": {"kind": "diet",
                   "guest": {"en": ["vegetarian", "no meat"], "ko": ["채식", "베지테리언"], "ja": ["ベジタリアン", "菜食"], "zh": ["素食", "吃素"]},
                   "host": {"tl": ("walang karne (vegetarian)", "high"), "ceb": ("walay karne (vegetarian)", "check"),
                            "ilo": ("awan ti karne (vegetarian)", "check")}},
    "vegan": {"kind": "diet",
              "guest": {"en": ["vegan"], "ko": ["비건", "완전 채식"], "ja": ["ヴィーガン", "ビーガン"], "zh": ["纯素", "純素"]},
              "host": {"tl": ("walang anumang galing sa hayop: karne, isda, itlog, gatas (vegan)", "high"),
                       "ceb": ("walay bisan unsa gikan sa hayop (vegan)", "check"), "ilo": ("awan ti aniaman a naggapu iti animal (vegan)", "check")}},
    "halal": {"kind": "diet",
              "guest": {"en": ["halal", "muslim"], "ko": ["할랄"], "ja": ["ハラール", "ハラル"], "zh": ["清真"]},
              "host": {"tl": ("halal: walang baboy at alak", "high"), "ceb": ("halal: walay baboy ug alak", "check"),
                       "ilo": ("halal: awan ti baboy ken arak", "check")}},
    "no_pork": {"kind": "diet",
                "guest": {"en": ["pork", "pig"], "ko": ["돼지고기"], "ja": ["豚肉", "ポーク"], "zh": ["猪肉", "豬肉"]},
                "host": {"tl": ("baboy", "high"), "ceb": ("baboy", "high"), "ilo": ("baboy", "high")}},
    "no_alcohol": {"kind": "diet",
                   "guest": {"en": ["alcohol", "no wine", "no beer"], "ko": ["술", "알코올"], "ja": ["お酒", "アルコール"], "zh": ["酒精", "不喝酒"]},
                   "host": {"tl": ("alak", "high"), "ceb": ("alak / ilimnon", "check"), "ilo": ("arak", "check")}},
    # ---- health, children, access, weather risk
    "allergy_general": {"kind": "health",
                        "guest": {"en": ["allerg", "anaphyla", "epipen"], "ko": ["알레르기", "알러지", "에피펜"], "ja": ["アレルギー", "エピペン"], "zh": ["过敏", "過敏"]},
                        "host": {"tl": ("may allergy (maaaring delikado)", "high"), "ceb": ("adunay alerdyi (mahimong delikado)", "check"),
                                 "ilo": ("adda alerdyi (mabalin a peligro)", "check")}},
    "medical": {"kind": "health",
                "guest": {"en": ["diabet", "asthma", "heart", "medicine", "pregnan", "sick", "hospital", "doctor"],
                          "ko": ["당뇨", "천식", "심장", "약", "임신", "아파", "병원", "의사"],
                          "ja": ["糖尿", "喘息", "心臓", "薬", "妊娠", "具合", "病院", "医者"],
                          "zh": ["糖尿", "哮喘", "心脏", "心臟", "药", "藥", "怀孕", "懷孕", "生病", "医院", "醫院", "医生"]},
                "host": {"tl": ("may kondisyong pangkalusugan", "high"), "ceb": ("adunay kondisyon sa panglawas", "check"),
                         "ilo": ("adda kondision ti salun-at", "check")}},
    "child": {"kind": "child",
              "guest": {"en": ["child", "kid", "son", "daughter", "baby", "toddler", "infant"], "ko": ["아이", "아들", "딸", "아기", "어린이"],
                        "ja": ["子供", "子ども", "息子", "娘", "赤ちゃん"], "zh": ["孩子", "儿子", "兒子", "女儿", "女兒", "婴儿", "嬰兒", "小孩"]},
              "host": {"tl": ("may kasamang bata", "high"), "ceb": ("adunay kauban nga bata", "check"), "ilo": ("adda kadua nga ubing", "check")}},
    "accessibility": {"kind": "access",
                      "guest": {"en": ["wheelchair", "disab", "walking stick", "cane", "elderly", "mobility"], "ko": ["휠체어", "장애", "지팡이", "노인"],
                                "ja": ["車椅子", "車いす", "障害", "杖", "高齢"], "zh": ["轮椅", "輪椅", "残疾", "殘疾", "拐杖", "老人"]},
                      "host": {"tl": ("kailangan ng tulong sa paglalakad / wheelchair", "high"),
                               "ceb": ("nagkinahanglan og tabang sa paglakaw / wheelchair", "check"),
                               "ilo": ("masapul ti tulong iti panagnagna / wheelchair", "check")}},
    "weather_risk": {"kind": "weather",
                     "guest": {"en": ["rain", "typhoon", "storm", "flood", "slippery", "heat", "landslide"],
                               "ko": ["비", "태풍", "폭풍", "홍수", "미끄", "더위", "산사태"],
                               "ja": ["雨", "台風", "嵐", "洪水", "滑り", "暑", "土砂"],
                               "zh": ["雨", "台风", "颱風", "暴风", "洪水", "滑", "炎热", "山体滑坡"]},
                     "host": {"tl": ("nagtatanong tungkol sa panganib dahil sa panahon", "high"),
                              "ceb": ("nangutana bahin sa peligro tungod sa panahon", "check"),
                              "ilo": ("agsaludsod maipanggep iti peligro gapu iti panawen", "check")}},
}
# Words whose very short CJK keywords cause false positives; matched only as whole tokens.
SHORT_KO = {"게", "밀", "콩", "깨", "술", "약", "비"}

CARD_TITLE = {"tl": "PAALALA SA KALIGTASAN", "ceb": "PAHINUMDOM SA KALUWASAN", "ilo": "PALAGIP MAIPANGGEP ITI KINATALGED"}
CARD_ASK = {"tl": "Pakisuri ang lahat ng sangkap bago ihain. Kung hindi sigurado, huwag ihain.",
            "ceb": "Palihug susiha ang tanang sangkap sa dili pa idalit. Kung dili sigurado, ayaw idalit.",
            "ilo": "Pangngaasiyo ta kitaenyo amin a ramen sakbay nga idasar. No saan a siguradokayo, dikay idasar."}
REVIEWED: dict[tuple[str, str], bool] = {}   # (item, host_lang) -> True once a native speaker approves


def _norm(t: str) -> str:
    return unicodedata.normalize("NFKC", t).lower()


def detect(text: str) -> list[str]:
    """Return item keys whose guest keywords appear in text, in any guest language."""
    t = _norm(text)
    tokens = set(re.findall(r"[\w']+", t))
    hits = []
    for key, item in ITEMS.items():
        for lang, words in item["guest"].items():
            for w in words:
                w = _norm(w)
                if lang == "en":
                    ok = re.search(r"\b" + re.escape(w), t) is not None
                elif lang == "ko" and w in SHORT_KO:
                    ok = any(tok == w or tok.startswith(w + "이") or tok.startswith(w + "는") or tok.startswith(w + "가")
                             or tok.startswith(w + "를") or tok.startswith(w + "에") for tok in tokens)
                else:
                    ok = w in t
                if ok:
                    hits.append(key)
                    break
            if hits and hits[-1] == key:
                break
    return hits


def card(hits: list[str], host_lang: str) -> str:
    """Fixed host-language safety card. Unreviewed terms are visibly marked."""
    lines = [CARD_TITLE[host_lang]]
    for key in hits:
        term, conf = ITEMS[key]["host"][host_lang]
        flag = "" if REVIEWED.get((key, host_lang)) else " [draft]"
        lines.append(f"- {term.upper()}{flag}")
    lines.append(CARD_ASK[host_lang])
    return "\n".join(lines)


def route(text: str, model_intent: str) -> str:
    """Safety override: any detector hit forces DIETARY_SAFETY."""
    return "DIETARY_SAFETY" if detect(text) else model_intent


def review_rows():
    """Rows for the native-speaker review sheet."""
    for key, item in ITEMS.items():
        for h, (term, conf) in item["host"].items():
            yield {"item": key, "kind": item["kind"], "host_lang": h, "draft_term": term,
                   "confidence": conf, "english_meaning": key.replace("_", " "),
                   "approved (yes/no)": "", "correction": "", "reviewer": ""}


## 3. Detector Evaluation

Recall is measured on hand-written safety messages in each guest language. False alarms are measured on two sources: hand-written ordinary requests, and FLORES+ news sentences, which are not about the farm but contain everyday vocabulary. A FLORES hit is not necessarily wrong (many sentences mention disease or weather); the rate shows how noisy the rules are.

In [ ]:
SAFETY_MSGS = {
 "en": ["My son is allergic to peanuts. Is the food safe for him?", "I can't eat shrimp or crab.",
        "Is the trail safe if it rains?", "My mother uses a wheelchair.", "We are vegetarian.",
        "Is the lunch halal?", "I am pregnant, is the walk too hard?", "Does the dish contain milk or egg?"],
 "ko": ["아들이 땅콩 알레르기가 있어요. 음식은 안전한가요?", "새우나 게를 못 먹어요.", "비가 오면 길이 안전한가요?",
        "어머니가 휠체어를 사용하세요.", "저희는 채식주의자예요.", "점심은 할랄인가요?"],
 "ja": ["息子はピーナッツアレルギーです。料理は大丈夫ですか？", "えびとかにが食べられません。", "雨が降ったら道は安全ですか？",
        "母は車椅子を使っています。", "私たちはベジタリアンです。", "ランチはハラールですか？"],
 "zh": ["我儿子对花生过敏，食物安全吗？", "我不能吃虾和蟹。", "下雨的话路安全吗？",
        "我妈妈坐轮椅。", "我们吃素。", "午餐是清真的吗？"]}
ORDINARY = {
 "en": ["How much is the farm tour per person?", "Can we book for four people on Saturday morning?",
        "How do we get to your farm from the bus terminal?", "What time does the tour start?", "Thank you so much!"],
 "ko": ["농장 투어는 1인당 얼마예요?", "토요일 아침에 4명 예약할 수 있나요?", "버스 터미널에서 농장까지 어떻게 가요?", "감사합니다!"],
 "ja": ["ツアーは一人いくらですか？", "土曜日の朝に4人で予約できますか？", "バスターミナルから農場までどう行きますか？", "ありがとうございます！"],
 "zh": ["农场游每人多少钱？", "周六早上可以预约四个人吗？", "从汽车站怎么去你的农场？", "非常感谢！"]}

rec = {l: sum(bool(detect(m)) for m in ms) / len(ms) for l, ms in SAFETY_MSGS.items()}
missed = {l: [m for m in ms if not detect(m)] for l, ms in SAFETY_MSGS.items()}
fa = {l: [m for m in ms if detect(m)] for l, ms in ORDINARY.items()}
print("Recall on safety messages:", rec); print("Missed:", missed); print("False alarms on ordinary requests:", fa)

from datasets import load_dataset, get_dataset_config_names
FLORES = "openlanguagedata/flores_plus"
configs = get_dataset_config_names(FLORES, token=HF_TOKEN)
FCFG = {"en": "eng_Latn", "ko": "kor_Hang", "ja": "jpn_Jpan", "zh": "cmn_Hans", "ceb": "ceb_Latn",
        "ilo": "ilo_Latn", "tl": next(c for c in configs if c.startswith(("tgl_", "fil_")))}
text = {l: {r["id"]: r["text"] for r in load_dataset(FLORES, c, split="devtest", token=HF_TOKEN)} for l, c in FCFG.items()}
flores_rate = {}
for l in ["en", "ko", "ja", "zh"]:
    hits = [detect(s) for s in text[l].values()]
    from collections import Counter
    flores_rate[l] = {"rate": round(sum(map(bool, hits)) / len(hits), 3),
                      "top_items": Counter(k for h in hits for k in h).most_common(5)}
print("Detector hit rate on FLORES news sentences:", flores_rate)
RESULTS["detector"] = {"recall": rec, "missed": missed, "false_alarms_ordinary": fa, "flores_hit_rate": flores_rate}
save()

## 4. Direct versus Pivot Translation

Direct: guest language to host language in one step. Pivot: guest language to English, then English to host language. Scores are chrF++ on the same FLORES+ devtest sentences.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
NLLB = {"en": "eng_Latn", "ko": "kor_Hang", "ja": "jpn_Jpan", "zh": "zho_Hans", "tl": "tgl_Latn", "ceb": "ceb_Latn", "ilo": "ilo_Latn"}
tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
mt = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", torch_dtype=torch.float16).to(DEVICE).eval()
def translate(sents, src, tgt, bs=16):
    tok.src_lang = NLLB[src]; out = []
    for i in range(0, len(sents), bs):
        x = tok(sents[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            y = mt.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB[tgt]), max_new_tokens=256, num_beams=4)
        out += tok.batch_decode(y, skip_special_tokens=True)
    return out

ids = sorted(set.intersection(*[set(v) for v in text.values()]))[:N_MT]
RESULTS["pivot"] = {}
for g in ["ko", "ja", "zh"]:
    src = [text[g][i] for i in ids]
    via_en = translate(src, g, "en")
    for h in ["tl", "ceb", "ilo"]:
        refs = [text[h][i] for i in ids]
        d = sacrebleu.corpus_chrf(translate(src, g, h), [refs], word_order=2).score
        p = sacrebleu.corpus_chrf(translate(via_en, "en", h), [refs], word_order=2).score
        RESULTS["pivot"][f"{g}->{h}"] = {"direct": round(d, 1), "pivot_en": round(p, 1), "gain": round(p - d, 1)}
        print(f"{g}->{h}: direct {d:.1f} | via English {p:.1f} | gain {p - d:+.1f}")
save()

## 5. Allergen Term Preservation

For each allergen sentence, we check whether the host-language output contains the expected word from the safety bank (for example `mani` for peanut). This is a direct measure of the failure seen in the first evaluation.

In [ ]:
ALLERGEN_TESTS = {  # item -> sentence per guest language
 "peanut": {"en": "My son is allergic to peanuts.", "ko": "아들이 땅콩 알레르기가 있어요.", "ja": "息子はピーナッツアレルギーです。", "zh": "我儿子对花生过敏。"},
 "egg": {"en": "I am allergic to eggs.", "ko": "저는 달걀 알레르기가 있어요.", "ja": "私は卵アレルギーです。", "zh": "我对鸡蛋过敏。"},
 "fish": {"en": "I cannot eat fish.", "ko": "저는 생선을 못 먹어요.", "ja": "私は魚が食べられません。", "zh": "我不能吃鱼。"},
 "no_pork": {"en": "We do not eat pork.", "ko": "저희는 돼지고기를 안 먹어요.", "ja": "私たちは豚肉を食べません。", "zh": "我们不吃猪肉。"},
 "milk": {"en": "My daughter cannot drink milk.", "ko": "딸은 우유를 못 마셔요.", "ja": "娘は牛乳が飲めません。", "zh": "我女儿不能喝牛奶。"},
}
def expected(item, h):  # first word of the bank term, lowercased
    return ITEMS[item]["host"][h][0].split()[0].split("/")[0].strip().lower()
rows = []
for item, sents in ALLERGEN_TESTS.items():
    for g, s in sents.items():
        for h in ["tl", "ceb", "ilo"]:
            direct = translate([s], g, h)[0]
            pivot = direct if g == "en" else translate(translate([s], g, "en"), "en", h)[0]
            e = expected(item, h)
            rows.append({"item": item, "guest": g, "host": h, "expected": e, "direct": direct, "pivot": pivot,
                         "direct_ok": e in direct.lower(), "pivot_ok": e in pivot.lower()})
import pandas as pd
df = pd.DataFrame(rows)
print("Allergen word preserved (share of sentences):")
print(df.groupby("host")[["direct_ok", "pivot_ok"]].mean().round(2))
print(df[~df.direct_ok | ~df.pivot_ok][["item", "guest", "host", "expected", "direct", "pivot"]].to_string())
RESULTS["allergen_preservation"] = {"by_host": df.groupby("host")[["direct_ok", "pivot_ok"]].mean().round(3).to_dict(),
                                    "overall": {"direct": round(df.direct_ok.mean(), 3), "pivot": round(df.pivot_ok.mean(), 3)},
                                    "rows": rows}
df.to_csv(OUT / "allergen_tests.csv", index=False)
save()

## 6. Example Safety Cards

What a host would see instead of a free translation. Terms marked `[draft]` are awaiting native-speaker review.

In [ ]:
for msg in ["我儿子对花生过敏，食物安全吗？", "えびとかにが食べられません。", "Is the trail safe if it rains?"]:
    hits = detect(msg)
    print("Guest:", msg, "| detected:", hits)
    for h in ["tl", "ceb", "ilo"]:
        print(card(hits, h)); print()
    print("-" * 60)

## 7. Package

In [ ]:
import shutil
from IPython.display import FileLink
shutil.make_archive("/kaggle/working/safety_out", "zip", OUT)
os.chdir("/kaggle/working"); FileLink("safety_out.zip")